<a href="https://colab.research.google.com/github/hanbiphyun/ESSA_YB/blob/main/ESAA_OB_week4_1_%EC%88%98%EC%83%81%EC%9E%91%EB%A6%AC%EB%B7%B0.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

링크 : https://www.kaggle.com/c/elo-merchant-category-recommendation
## Kaggle 수상작 리뷰: Elo Merchant Category Recommendation (1st Place)

### 1. 대회 개요 및 목표
* **주제:** 고객의 과거 거래 데이터를 바탕으로 미래의 고객 충성도 점수 예측
* **평가 지표:** RMSE
* **핵심 과제:** 수천만 건의 거래 내역에서 유의미한 피처를 추출하고, 특히 점수가 튀는 아웃라이어를 어떻게 처리할 것인가가 중요했음.

### 2. 주요 전략

#### ① 강력한 피처 엔지니어링
단순한 통계량(평균, 합계)을 넘어, **잠재 요인**을 추출하기 위한 기법이 핵심적으로 사용되었습니다.
* **NLP 기법의 전용:** 고객이 방문한 상점 ID를 하나의 '단어'로 간주하고 `Word2Vec`을 적용하여 상점과 고객의 잠재적인 관계(임베딩)를 벡터화함.
* **행렬 분해(Matrix Factorization):** 고객-상점 행렬을 분해하여 잠재 요인을 피처로 활용.

#### ② 아웃라이어 처리를 위한 이중 모델링
데이터 중 약 1%를 차지하는 극단적인 아웃라이어가 RMSE를 크게 망가뜨리는 문제가 있었습니다.
* **Classification + Regression:** 먼저 해당 데이터가 아웃라이어인지 아닌지 분류하는 모델을 만들고, 그 확률값을 가중치로 사용하여 회귀를 진행함.

#### ③ 모델 스택킹 (Model Stacking)
단일 모델의 한계를 극복하기 위해 `LightGBM`, `XGBoost`, `CatBoost`를 기본 모델로 사용하고, 최종적으로 이들을 결합하는 **Stacking** 기법을 적용하여 1위를 차지함.

### 3. 핵심 코드 분석

In [ ]:
# [피처 생성: 데이터의 특징을 요약하는 과정]
import pandas as pd
import numpy as np

# 거래 데이터에서 고객별 통계량(Latent Insights) 추출
def aggregate_transactions(trans):
    agg_func = {
        'purchase_amount': ['sum', 'mean', 'max', 'min', 'std'],
        'installments': ['sum', 'mean', 'max', 'min', 'std'],
        'merchant_id': ['nunique'],
        'month_lag': ['mean', 'std']
    }
    agg_trans = trans.groupby(['card_id']).agg(agg_func)

    # 컬럼 이름 재설정
    agg_trans.columns = [ '_'.join(col).strip() for col in agg_trans.columns.values]
    agg_trans.reset_index(inplace=True)
    return agg_trans

In [ ]:
# [모델링: LightGBM을 활용한 예측]

import lightgbm as lgb

# 하이퍼파라미터 설정 (우승팀들이 선호하는 설정)
lgb_params = {
    'objective': 'regression',
    'metric': 'rmse',
    'boosting_type': 'gbdt',
    'learning_rate': 0.01,
    'feature_fraction': 0.9,
    'bagging_fraction': 0.7,
    'num_leaves': 31,
    'seed': 42
}

# 학습 데이터 준비 및 교차 검증(K-Fold)
# dtrain = lgb.Dataset(X_train, label=y_train)
# model = lgb.train(lgb_params, dtrain, num_boost_round=10000)

### 4. 시사점 및 배운 점
* **도메인 지식의 중요성:** 단순한 모델링보다 금융 거래 데이터의 흐름(구매 주기, 할부 비율 등)을 이해하고 피처를 만든 것이 주효했음.
* **잠재 요인의 영향력:** 정형 데이터에서도 `SVD`나 `Word2Vec` 같은 비정형 데이터 분석 기법을 통해 숨겨진 특징(잠재요인)을 찾아내는 것이 예측 성능을 극적으로 높임.
* **검증의 엄밀함:** 자신만의 교차 검증전략을 세우는 것이 중요했음.
